In [1]:
# import from main and experiments library
import os
from exp_library import *
os.chdir("../")
from library import *

# filter the warnings for clarity
import warnings
warnings.filterwarnings("ignore")

## Bankruptcy Prediction - Numerical Models 💸📜📈

### Introduction

In this notebook, we include the experiments for the next-year bankruptcy prediction task using numerical predictors from the following paper:

```... add citation ...```

### The Task

Before we dive into the models, let's look at the prediction task. 10K filings are comprehensive documents submitted annually by companies to the U.S. Securities and Exchange Commission (SEC). These filings cover a fiscal year, which concludes on the fiscal year end. After the filing period, which ends on the filing date, the reports are released to the public and they can be accessed by any interested stakeholder. The data in the reports is multimodal (numerical and textual) and our aim is to develop bankruptcy prediction models using this data. More specifically, the models predict whether the company will file for bankruptcy in the year following the filing date, using the data contained in the 10K. For more details on the prediction setup, we refer to the paper above.

The ECL dataset contains almost 85,000 labelled instances that can be used to train and evaluate the models. To extract the subset of labelled 10K filings, use the following variables:

- **qualified (str):** 'Yes' if the 10K record qualifies for inclusion in the LoPucki BRD, 'No' if the 10K record does not qualify for inclusion in the LoPucki BRD and 'out-of-period' if the 10K records was filed before 1993 or after 2021.
- **can_label (bool):** True if we have all the necessary information to assign a label to the 10K record, False otherwise (i.e. we know the filing date and the total asset value reported in the 10K).
- **label (bool):** True if the company filed for bankruptcy in the year following the filing date of the 10K, False otherwise.

```The prediction task is visually shown in the figure below:```

<div>
<img src="../images/task.png" width="800" align="left"/>
</div>

### Read in the data

In [3]:
# Read in the dataset.
dataset = pd.read_csv('ECL.csv', index_col=0)

### Add numerical predictors

#### Option 1: Extract CompuStat data via the WRDS Python API

In [ ]:
# Login using WRDS credentials.
username = ''
db = wrds.Connection(wrds_username=username)

# List the required variables
variables = 'act, lct, ap, sale, che, lt, mib, ch, ebit, dp, dlc, dltt, invch, invt, ni, oiadp, re, seq, wcap'

# Add the CompuStat variables to the MultiModalFinance dataset.
dataset = get_CompuStat_WRDS(variables, dataset, db)

#### Option 2: Extract CompuStat data via a local copy

In [4]:
# Path to the CompuStat data.
path = './data/CompuStat/data.csv'

# Add the CompuStat variables to the MultiModalFinance dataset.
dataset = get_CompuStat_local(path, dataset, update=False)

Dropped 115373 rows from CompuStat based on screening variables
0 records in the dataset do not have an accompanying CompuStat record.


#### Create train and test set

In [5]:
# Compute features and store the column indices.
dataset, predictors = compute_features(dataset)

In [6]:
# Store the records that can be labelled and are qualified for inclusion in the LoPucki BRD.
prediction_subset = dataset.loc[(dataset['can_label'] == True) & (dataset['qualified'] == 'Yes')].reset_index(drop=True)

# Split the data in a train and test set.
train_full = prediction_subset.loc[prediction_subset['bankruptcy_prediction_split'] == 'train']
test = prediction_subset.loc[prediction_subset['bankruptcy_prediction_split'] == 'test']

In [7]:
# Store predictors and labels
X = train_full[predictors]
y = train_full['label']

test_X = test[predictors]
test_y = test['label']

In [8]:
# Resample training data and store in a dictionary
training_data = dict()

# add orignal data distribution
training_data['real'] = (X, y)

# loop over the required data distributions
for i in [1, 0.5, 0.25]:
    
    # resample
    ros = RandomOverSampler(random_state=0, sampling_strategy=i)
    X_resampled, y_resampled = ros.fit_resample(X, y)
    
    # add to dictionary
    training_data[i] = (X_resampled, y_resampled)

# Models

In the following section, we show how we trained the models, after hyperparameter optimisation, using the sci-kit learn framework. We do not include the optimisation itself, but immediatly train the models with the optimal hyperparameter settings.

## Logistic regression

In [9]:
# create the pipeline
distribution = 1

LR = Pipeline([('imputer', SimpleImputer(missing_values=np.nan, strategy='mean')), 
                 ('scaler', StandardScaler()), 
                 ('clf', LogisticRegression(penalty='l2', C = 0.01))])

# train model
LR.fit(X=training_data[distribution][0], y=training_data[distribution][1])

# evaluate the model
preds = LR.predict_proba(test_X)[:, 1]
evaluate(labels=test_y, predictions=preds)

-- RESULTS --
AUC: 0.9148
AP: 0.115
recall@100: 0.1475
CAP: 0.8297


## MLP

In [10]:
# create the pipeline
distribution = 0.5

# note that the output of the MLP depends on the initial weights - the results will slightly vary each run
MLP = Pipeline([('imputer', SimpleImputer(missing_values=np.nan, strategy='mean')), 
                         ('scaler', StandardScaler()), 
                         ('clf', MLPClassifier(learning_rate='invscaling', alpha=1, learning_rate_init=0.001,
                                              hidden_layer_sizes=(100,100)))])
# train model
MLP.fit(X=training_data[distribution][0], y=training_data[distribution][1])

# evaluate the model
preds = MLP.predict_proba(test_X)[:, 1]
evaluate(labels=test_y, predictions=preds)

-- RESULTS --
AUC: 0.9227
AP: 0.1791
recall@100: 0.2049
CAP: 0.8454


## XGBoost

In [11]:
# create the pipeline
distribution = 1

XGB = Pipeline([ ('scaler', StandardScaler()), 
                 ('clf', xgb.XGBClassifier(objective='binary:logistic', subsample=0.5, eta=0.1, 
                 max_depth = 1, n_estimators = 1000))])

# train model
XGB.fit(X=training_data[distribution][0], y=training_data[distribution][1])

# evaluate the model
preds = XGB.predict_proba(test_X)[:, 1]
evaluate(labels=test_y, predictions=preds)

-- RESULTS --
AUC: 0.9364
AP: 0.1562
recall@100: 0.1885
CAP: 0.8727
